# 41장 · 화면 상태와 Graph 대화를 연결하기

한 학생이 대출 기간을 묻고 이어서 자료에 없는 주차 요금을 묻습니다. 기록 화면에는 이전 대화가 남아도 지금 답의 출처에는 대출 문서가 남으면 안 됩니다. 화면에 둘 값과 Graph에 넘길 값을 나누어 살펴봅니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 화면의 대화와 Graph 저장 위치 연결하기

실제 Graph에 InMemorySaver를 붙여 대화별로 고정 질문을 기록합니다. 화면 담당 코드는 현재 대화의 thread_id를 전달합니다. 여기서는 자료 등록 과정이 끝난 뒤의 가상 도서관 고정 자료를 쓰며 새 문서 업로드나 재색인은 실행하지 않습니다.



**볼 곳**　‘thread_id’를 기준으로 양쪽의 ‘화면 대화’와 ‘그래프 상태’를 대조하세요.

**애니메이션의 의미**　화면 대화를 그래프의 대화 ID와 연결합니다.


### 셀 01 · 제공 셀 · 검색 결과와 기록을 저장하는 그래프

history에는 누적 규칙을 붙이고 sources에는 현재 결과만 덮어쓰도록 합니다.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class Session(TypedDict):
    question: str
    sources: list[dict]
    history: Annotated[list[str], add]

def process(state):
    return {'sources': retrieve(state['question'], 1), 'history': [state['question']]}

builder = StateGraph(Session)
builder.add_node('process', process)
builder.add_edge(START, 'process')
builder.add_edge('process', END)
engine = builder.compile(checkpointer=InMemorySaver())

def send(thread, question):
    return engine.invoke({'question': question}, {'configurable': {'thread_id': thread}})

first = send('A', '대출 기간은?')
print(first['sources'][0]['id'])

library-1


A의 현재 결과는 library-1입니다. 반복 실행으로 새 엔진을 만들면 메모리 저장소도 다시 만들어집니다.

### 셀 02 · 같은 대화에서 현재 출처 비우기

A가 주차 질문을 보낸 직후의 기록과 출처를 읽으세요.

In [2]:
second = send('A', '주차 요금은?')
print(second['history'])
print(second['sources'])

['대출 기간은?', '주차 요금은?']
[]


질문 두 개가 기록에 있고 현재 출처는 비었습니다. 프런트엔드가 첫 답의 출처를 계속 들고 있다면 화면만 잘못된 상태일 수 있습니다.

## 사용자와 대화 사이의 경계 확인하기

화면의 새 대화 버튼은 새 ID와 빈 현재 화면을 준비합니다. 다른 사용자의 thread_id를 재사용하면 이전 상태를 이어 읽으므로 이 노트북의 A와 B 같은 예시 ID를 운영 인증에 쓰면 안 됩니다.



**볼 곳**　‘저장 공간’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘기존 대화’, 아래쪽은 ‘새 대화’입니다.

**애니메이션의 의미**　다른 세션에서 앞 사용자의 기록을 읽지 않도록 합니다.


### 과제 03 · 두 번째 대화를 열고 기록 확인하기

B라는 새 대화에 휴관일을 묻고 B의 질문 목록과 현재 쪽수를 표시하세요.

In [3]:
# 다른 ID로 send를 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = send(&#x27;B&#x27;, &#x27;휴관일은?&#x27;)
print(other[&#x27;history&#x27;], other[&#x27;sources&#x27;][0][&#x27;page&#x27;])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;휴관일은?&#x27;] 2</code></pre>
<p>B에는 휴관 질문 한 개와 2쪽만 보입니다. A의 두 질문이 나타났다면 호출한 ID와 화면에 남긴 기록을 각각 살펴보세요.</p>
</details>

## Streamlit의 입력을 Graph에 넘기기

제공 앱은 같은 검색 함수와 체크포인터를 씁니다. 앞서 본 처리 중 UI에서 결정하는 것은 대화 ID와 현재 표시할 결과입니다. 새 대화 버튼은 새 ID를 만듭니다. 수업용 예제에는 로그인·운영 저장소가 없습니다.



**볼 곳**　‘사용자 입력’에서 출발해 ‘그래프 실행’를 거친 뒤 ‘답변 화면’가 되는 순서를 따라가세요.

**애니메이션의 의미**　화면 입력을 그래프에 보내고 응답을 다시 표시합니다.


### 셀 04 · 제공 셀 · 대화 ID와 현재 화면 연결하기

st.session_state.thread를 설정하는 줄과 engine.invoke에 같은 ID를 건네는 줄을 찾아보세요.

노트북의 앱은 cache_resource로 엔진을 재사용합니다. A에서 B로 전환하면 엔진이 쓰는 저장소는 남지만 현재 화면은 B의 대화 ID와 출처를 기준으로 다시 그립니다. 뒤의 참고 앱은 session_state에 엔진을 두는 별도 구현입니다. 먼저 cache_resource로 그래프를 준비하는 줄과, 제출 뒤 현재 출처를 갱신하는 줄을 찾으세요.

In [4]:
ui_source = """records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import TypedDict
from uuid import uuid4
import streamlit as st
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated
from operator import add

class Chat(TypedDict):
    question: str
    sources: list[dict]
    history: Annotated[list[str], add]

def process(state):
    return {"sources": retrieve(state["question"], 1), "history": [state["question"]]}

@st.cache_resource
def prepare_engine():
    builder = StateGraph(Chat)
    builder.add_node("process", process)
    builder.add_edge(START, "process")
    builder.add_edge("process", END)
    return builder.compile(checkpointer=InMemorySaver())

new_chat = st.button("새 대화")
if "thread" not in st.session_state or new_chat:
    st.session_state.thread = str(uuid4())
    st.session_state.current = None
engine = prepare_engine()
with st.form("ask"):
    question = st.text_input("질문")
    submit = st.form_submit_button("질문 보내기")
if submit and question.strip():
    config = {"configurable": {"thread_id": st.session_state.thread}}
    st.session_state.current = engine.invoke({"question": question}, config)
if st.session_state.current:
    state = st.session_state.current
    st.text("지난 질문: " + " / ".join(state["history"]))
    if state["sources"]:
        current = state["sources"][0]
        st.write(current["text"])
        st.caption(f"근거 [{current['id']}] {current['page']}쪽")
    else:
        st.info("제공된 자료에서 확인할 수 없습니다.")
"""
import ast
ast.parse(ui_source)
print("앱 구문 확인")

앱 구문 확인


구문 확인 뒤 다음 셀에서 제출 이벤트를 보냅니다. prepare_engine은 위젯을 조작할 때마다 저장소를 새로 만들지 않도록 엔진을 캐시합니다. 수업용 화면 주소가 있으면 그 주소를 여세요. 직접 열 때는 이 셀 아래의 파일을 받은 뒤 수업 환경에서 python -m streamlit run chapter41_form.py 명령을 실행하세요. 같은 대화에서 대출 질문 다음에 주차 질문을 제출하고 이전 출처가 사라지는지 확인하세요.

<p><a download="chapter41_form.py" href="">셀에 나온 Streamlit 화면 파일 받기</a></p>

### 셀 05 · 같은 폼에서 두 질문 제출하기

Streamlit 커널이면 AppTest로 대출 질문 다음 주차 질문을 보냅니다. 최신 출처가 사라지고 이전 질문 기록이 남는지 읽으세요.

In [5]:
from importlib.util import find_spec
if find_spec("streamlit") is None:
    print("Streamlit이 설치된 수업 커널에서 화면 연결을 확인하세요.")
else:
    from streamlit.testing.v1 import AppTest
    ui = AppTest.from_string(ui_source).run()
    ui.text_input[0].set_value("대출 기간은?")
    ui.button[1].click().run()
    assert not ui.exception and ui.caption[0].value.endswith("1쪽")
    ui.text_input[0].set_value("주차 요금은?")
    ui.button[1].click().run()
    assert not ui.exception and not ui.caption
    print(ui.text[0].value, ui.info[0].value)

Streamlit이 설치된 수업 커널에서 화면 연결을 확인하세요.


설치된 환경에서는 지난 질문 두 개와 자료 없음 안내가 보여야 합니다. 설치 안내만 출력되면 위젯 실습은 아직 실행하지 않은 상태입니다. 앞 절의 Graph 실습은 별도로 확인할 수 있습니다.

## 돌아보기

문서 연결이 실패하거나 검색 오류가 발생한 경우에는 현재 답과 출처를 먼저 지우세요. 사용자가 지난 질문을 다시 읽을 수 있도록 과거 기록은 분리해 둡니다.